In [1]:
# Simple LSTM for Next Word Prediction - Beginner Friendly

# Cell 1: Get Our Tools Ready
# ---------------------------
# We need some tools (libraries) to help us build our word predictor.
# - numpy is for handling numbers.
# - tensorflow and keras are for building the "brain" (neural network).
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import string # To help remove punctuation like commas and periods.

print("TensorFlow Version:", tf.__version__)

# Cell 2: Our Sample Text
# -----------------------
# This is the text our computer will learn from.
# Let's use a few very simple sentences.
corpus = [
    "hello world",
    "good morning world",
    "hello there",
    "how are you",
    "i am fine thank you",
    "thank you good bye",
    "see you later"
]

print("Our simple sentences (Corpus):")
for sentence in corpus:
    print(sentence)

# Cell 3: Clean Up The Text & Break It Into Words (Tokenize)
# ----------------------------------------------------------
# 1. Make all words lowercase (e.g., "Hello" becomes "hello").
# 2. Remove punctuation (like '!' or '?').
# 3. Give each unique word a special number (this is called tokenization).

# Function to clean text
def clean_text_simple(text_lines):
    cleaned_lines = []
    # This translator will remove punctuation
    translator = str.maketrans('', '', string.punctuation)
    for line in text_lines:
        line = line.lower() # Make lowercase
        line = line.translate(translator) # Remove punctuation
        cleaned_lines.append(line)
    return cleaned_lines

cleaned_corpus = clean_text_simple(corpus)
print("\nCleaned Sentences:")
for sentence in cleaned_corpus:
    print(sentence)

# Tokenizer: Turns words into numbers
# <unk> is for words it hasn't seen before (Out Of Vocabulary)
tokenizer = Tokenizer(oov_token="<unk>")
tokenizer.fit_on_texts(cleaned_corpus) # Learns all the unique words

# How many unique words do we have? (+1 for the <unk> token)
vocab_size = len(tokenizer.word_index) + 1
print(f"\nNumber of unique words (Vocabulary Size): {vocab_size}")
print("Words and their assigned numbers (Word Index):")
# Let's see the first 5 words and their numbers
for i, (word, index) in enumerate(tokenizer.word_index.items()):
    if i < 10: # Print up to 10 words from the vocabulary
        print(f"'{word}': {index}")
    else:
        break


# Cell 4: Create Training Examples
# --------------------------------
# We need to create pairs of (input words) -> (next word to predict).
# Example: from "hello world there"
#   - "hello" -> "world"
#   - "hello world" -> "there"

input_sequences = []
for line in cleaned_corpus:
    # Turn the sentence into a list of numbers
    token_list = tokenizer.texts_to_sequences([line])[0]
    # Create sequences from the token list
    for i in range(1, len(token_list)):
        # Take a part of the sentence, from start up to word i
        # e.g., if token_list is [1, 2, 3] (for "hello world there")
        # i=1: n_gram_sequence = [1, 2] ("hello world")
        # i=2: n_gram_sequence = [1, 2, 3] ("hello world there")
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

if not input_sequences:
    raise ValueError("No sequences were made. Is the corpus too short or empty?")

# print("\nSample sequences (numbers):")
# for i in range(min(5, len(input_sequences))): # Show first 5
#    print(input_sequences[i])

# Make all sequences the same length by adding zeros at the beginning (padding)
# Find the longest sequence first
max_sequence_len = 0
if input_sequences: # Check if list is not empty
    max_sequence_len = max([len(x) for x in input_sequences])
else:
    print("Warning: No input sequences generated. Cannot determine max_sequence_len.")
    # Default to a small number if no sequences, though this indicates an issue
    max_sequence_len = 3 # Arbitrary small number, training will likely fail

if max_sequence_len <= 1 and input_sequences: # Need at least 2 for X and y
     raise ValueError(f"Max sequence length is {max_sequence_len}. "
                      "This is too short. Corpus sentences might be too short (e.g. single words). "
                      "Need sequences of at least length 2 to create X and y pairs.")


# Pad sequences
# 'pre' means add zeros at the beginning
padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

# print("\nSample padded sequences (numbers):")
# for i in range(min(5, len(padded_sequences))): # Show first 5
#    print(padded_sequences[i])


# Split into input (X) and target (y)
# X = all words except the last one
# y = the very last word (this is what we want to predict)
# Example: if padded_sequence is [0, 1, 2, 3] ("<pad> hello world there")
# X = [0, 1, 2] ("<pad> hello world")
# labels (target word number) = 3 ("there")
X = padded_sequences[:, :-1]
labels = padded_sequences[:, -1]

# Convert the target 'labels' into a special format (one-hot encoding)
# This is like saying: if vocab_size is 10 and word is 3, y will be [0,0,1,0,0,0,0,0,0,0]
y = to_categorical(labels, num_classes=vocab_size)

# The length of our input sequences for the model
# This is max_sequence_len - 1 because we took one word off for the 'y' (target)
model_input_length = max_sequence_len - 1
if model_input_length <= 0:
    raise ValueError(f"Model input length is {model_input_length}. This is too short. "
                     "Check corpus and max_sequence_len. Sentences might be too short.")


print(f"\nShape of X (our inputs): {X.shape}") # (number_of_sequences, length_of_each_sequence)
print(f"Shape of y (our targets, one-hot): {y.shape}") # (number_of_sequences, number_of_unique_words)
print(f"Length of sequences going into the model: {model_input_length}")


# Cell 5: Build The "Brain" (LSTM Model)
# --------------------------------------
# 1. Embedding Layer: Learns a nice representation for each word number.
# 2. LSTM Layer: The main part that understands sequences (like sentences).
# 3. Dense Layer: The output layer that predicts the next word.

embedding_dim = 20  # How much detail to store for each word's meaning (smaller for simplicity)
lstm_units = 30     # How many "memory cells" in our LSTM (smaller for simplicity)

model = Sequential()
# Layer 1: Embedding (turns word numbers into meaningful vectors)
model.add(Embedding(input_dim=vocab_size,         # Number of unique words
                    output_dim=embedding_dim,     # Size of the word vector
                    input_length=model_input_length)) # Length of our input sequences (X)

# Layer 2: LSTM (processes the sequence of word vectors)
model.add(LSTM(units=lstm_units))

# Layer 3: Dense Output Layer (predicts the next word from all possible words)
model.add(Dense(units=vocab_size, activation='softmax')) # Softmax gives probabilities for each word

# Get the model ready for training
model.compile(optimizer='adam',             # How the model learns (Adam is a good default)
              loss='categorical_crossentropy', # How we measure error for this type of problem
              metrics=['accuracy'])         # What we want to track (how often it's right)

model.summary() # Print a summary of our model

# Cell 6: Teach The Model (Training)
# ----------------------------------
# We show the model our X (inputs) and y (targets) many times.
# `epochs`: How many times to go through all the data.
# `batch_size`: How many examples to look at before updating the model's brain.

# Check if X or y is empty before training
if X.shape[0] == 0 or y.shape[0] == 0:
    raise ValueError("Input data X or labels y is empty. Cannot train. "
                     "This can happen if your corpus is too small or sentences are very short.")
elif X.shape[0] != y.shape[0]:
    raise ValueError(f"X and y have different number of samples: X={X.shape[0]}, y={y.shape[0]}.")


print(f"\nStarting training with {X.shape[0]} examples...")
try:
    # Let's train for fewer epochs for this simple example
    history = model.fit(X, y, epochs=50, batch_size=8, verbose=1)
    print("\nTraining finished.")
except Exception as e:
    print(f"Error during training: {e}")
    print("Is your dataset very, very small? Or are sequence lengths an issue?")


# Cell 7: Function to Predict Next Words
# --------------------------------------
# This function will take some starting words (seed_text)
# and use our trained model to guess the next few words.

def predict_next_words_simple(seed_text, num_next_words, model, tokenizer, current_model_input_length):
    output_text = seed_text
    current_text = seed_text.lower() # Match training data format

    for _ in range(num_next_words):
        # Turn current text into numbers
        token_list = tokenizer.texts_to_sequences([current_text])[0]
        if not token_list:
            print(f"Warning: Seed '{current_text}' has no known words. Stopping.")
            break

        # Pad the numbers to the correct length for the model
        padded_token_list = pad_sequences([token_list], maxlen=current_model_input_length, padding='pre')

        if padded_token_list.shape[1] == 0 and current_model_input_length > 0 :
             print(f"Warning: Padded token list for '{current_text}' became empty. "
                   f"This can happen if seed text is shorter than padding length and contains unknown words. "
                   f"Model input length expected: {current_model_input_length}. Seed token list: {token_list}")
             break
        if padded_token_list.shape[1] != current_model_input_length:
            print(f"Warning: Padded token list shape mismatch. Expected length {current_model_input_length}, got {padded_token_list.shape[1]}. Seed: '{current_text}'")
            # This might happen if all words in seed are <unk> and then padding makes it too short or wrong.
            # Or if current_model_input_length is 0.
            break


        # Ask the model to predict
        predicted_probabilities = model.predict(padded_token_list, verbose=0)[0]

        # Find the word number with the highest probability
        predicted_index = np.argmax(predicted_probabilities)

        # Turn the number back into a word
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break

        if not output_word: # Should not happen if vocab_size and model are correct
            print(f"Warning: Could not find word for predicted index {predicted_index}.")
            break

        # Add the predicted word to our text
        current_text += " " + output_word
        output_text += " " + output_word

    return output_text

# Cell 8: Test Our Predictor
# --------------------------
# Let's see what our model predicts!
if 'history' in locals() and history is not None: # Check if training actually ran
    seed1 = "hello"
    num_predict = 3

    # Important: model_input_length is X.shape[1], which is max_sequence_len - 1
    if model_input_length <= 0:
        print("Error: Model input length is not positive. Cannot predict.")
    else:
        prediction1 = predict_next_words_simple(seed1, num_predict, model, tokenizer, model_input_length)
        print(f"\nStart: '{seed1}'")
        print(f"Prediction: '{prediction1}'")

    seed2 = "how are"
    prediction2 = predict_next_words_simple(seed2, num_predict, model, tokenizer, model_input_length)
    print(f"\nStart: '{seed2}'")
    print(f"Prediction: '{prediction2}'")

    seed3 = "thank you"
    prediction3 = predict_next_words_simple(seed3, num_predict, model, tokenizer, model_input_length)
    print(f"\nStart: '{seed3}'")
    print(f"Prediction: '{prediction3}'")
else:
    print("\nModel training didn't complete. Skipping prediction test.")


# Cell 9: What's Next? (Simple Ideas)
# -----------------------------------
# 1. More Data: The more good examples you give the computer, the better it learns.
#    Try adding more sentences to the `corpus` in Cell 2.
#
# 2. Train Longer: In Cell 6, you can try increasing `epochs` (e.g., to 100 or 200).
#    This gives the model more time to learn.
#
# 3. Bigger Brain: In Cell 5, you can try slightly larger numbers for
#    `embedding_dim` (e.g., 50) or `lstm_units` (e.g., 50).
#    This gives the model more capacity to learn complex patterns.
#
# (Remember, with a very small dataset like this, predictions might still be a bit random or repetitive!)

print("\n--- End of Simple LSTM Next Word Prediction ---")


TensorFlow Version: 2.18.0
Our simple sentences (Corpus):
hello world
good morning world
hello there
how are you
i am fine thank you
thank you good bye
see you later

Cleaned Sentences:
hello world
good morning world
hello there
how are you
i am fine thank you
thank you good bye
see you later

Number of unique words (Vocabulary Size): 17
Words and their assigned numbers (Word Index):
'<unk>': 1
'you': 2
'hello': 3
'world': 4
'good': 5
'thank': 6
'morning': 7
'there': 8
'how': 9
'are': 10

Shape of X (our inputs): (15, 4)
Shape of y (our targets, one-hot): (15, 17)
Length of sequences going into the model: 4


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Starting training with 15 examples...
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.0000e+00 - loss: 2.8305
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.1750 - loss: 2.8264
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.1333 - loss: 2.8226    
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2167 - loss: 2.8168 
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2167 - loss: 2.8131 
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2167 - loss: 2.8076 
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.2167 - loss: 2.8014
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2167 - loss: 2.8000
Epoch 9/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.2167 - loss: 2.7899
Epoch 10/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.3028 - loss: 2.7825
Epoch 11/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3056 - loss: 2.7835
Epoch 12/50
2/2 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# Cell 2: Our Sample Text - Reading from a file
# ---------------------------------------------
file_path = 'my_dataset.txt' # Put the name of your text file here
corpus = []

try:
    with open(file_path, 'r', encoding='utf-8') as file:
        # Read the whole file content
        full_text = file.read()
        # Split the text into lines (or sentences if your file is structured that way)
        corpus = full_text.split('\n') # Or use '.' to split by sentences, then clean up

    # Optional: Remove any empty lines
    corpus = [line for line in corpus if line.strip()]

    print(f"Read {len(corpus)} lines from {file_path}.")
    print("First 5 lines from the file:")
    for i in range(min(5, len(corpus))):
        print(corpus[i])

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
    corpus = ["default sentence if file not found", "another default sentence"] # Fallback

except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    corpus = ["error reading file", "please check data"] # Fallback